<a href="https://colab.research.google.com/github/santannalorena936/Computa-o-25.2-/blob/main/CodigoFonte_Assistencia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [106]:
streamlit_code = """
# ================= IMPORTAÇÕES =================
import streamlit as st
import sqlite3
from datetime import datetime
import os
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas
import pandas as pd
import io
import textwrap # Adicionado para corrigir problemas de indentação em strings multi-linha

# ================= BANCO =================
conn = sqlite3.connect("ordens.db", check_same_thread=False)
cursor = conn.cursor()

cursor.execute(textwrap.dedent('''\
    CREATE TABLE IF NOT EXISTS ordens (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        loja TEXT,
        cliente TEXT,
        equipamento TEXT,
        marca TEXT,
        modelo TEXT,
        defeito TEXT,
        data_entrada TEXT,
        data_reparo TEXT,
        mes_reparo TEXT
    )
'''))
conn.commit()

# ================= FUNÇÕES =================
def finalizar_os(os_id):
    data = datetime.now()

    data_reparo = data.strftime("%d/%m/%Y")
    MESES = [
    "JANEIRO","FEVEREIRO","MARÇO","ABRIL","MAIO","JUNHO",
    "JULHO","AGOSTO","SETEMBRO","OUTUBRO","NOVEMBRO","DEZEMBRO"
    ]

    mes_reparo = MESES[data.month - 1]

    try:
        cursor.execute(textwrap.dedent('''
                UPDATE ordens
                SET data_reparo=?, mes_reparo=?
                WHERE id=?
            '''),
            (data_reparo, mes_reparo, os_id)
        )

        conn.commit()

        return True

    except Exception as e:
        print("Erro ao finalizar OS:", e)
        return False

def gerar_pdf(os_data):
    if not os.path.exists("pdf"):
        os.makedirs("pdf")

    nome = f"pdf/OS_{os_data[0]}.pdf"
    c = canvas.Canvas(nome, pagesize=A4)

    c.setFont("Times-Roman", 14)

    c.drawString(200, 800, f"ORDEM DE SERVIÇO Nº {os_data[0]}")
    c.drawString(50, 750, f"LOJA: {str(os_data[1]).upper()}")
    c.drawString(50, 730, f"CLIENTE: {str(os_data[2]).upper()}")
    c.drawString(50, 710, f"EQUIPAMENTO: {str(os_data[3]).upper()}")
    c.drawString(50, 690, f"MARCA: {str(os_data[4]).upper()}")
    c.drawString(50, 670, f"MODELO: {str(os_data[5]).upper()}")
    c.drawString(50, 650, f"DEFEITO: {str(os_data[6]).upper()}")

    c.drawString(50, 630, f"Entrada: {os_data[7]}")
    c.drawString(50, 610, f"Reparo: {os_data[8] or 'Em andamento'}")

    c.save()
    return nome

# ================= INTERFACE =================
st.title("Sistema de Gestão de Manutenção – Belessa")

# Lista de lojas
LOJAS = [
    "Belessa Alagoinhas","Belessa Aracaju Centro","Belessa Aracaju Jardins",
    "Belessa Arapiraca","Belessa Av. Sete","Belessa Barreiras",
    "Belessa Boca do Rio","Belessa Cajazeiras","Belessa Camaçari",
    "Belessa Candeias","Belessa Cruz das Almas","Belessa Dias D'Ávila",
    "Belessa Eunapolis","Belessa Feira de Santana","Belessa Ilheus",
    "Belessa Itabaiana","Belessa Itabuna","Belessa Itaigara",
    "Belessa Itapuã","Belessa Jequie","Belessa Juazeiro",
    "Belessa Lagarto","Belessa Lauro de Freitas","Belessa Maceió Centro",
    "Belessa Maceió Jatiuca","Belessa Pau da Lima","Belessa Paulo Afonso",
    "Belessa Periperi","Belessa Porto Seguro","Belessa SAJ",
    "Belessa Simões Filho","Belessa Socorro","Belessa Uruguai",
    "Belessa Valença","Belessa Vitoria da Conquista","Loja não identificada"
]

# Lista de equipamentos
EQUIPAMENTOS = ["Motor porquinho", "Cabine", "Coletor", "Luminária", "Outros"]

# Lista de marcas
MARCAS = ["Motor porquinho", "Cabine", "Coletor", "Luminária", "Outros"]

# Lista de modelos
MODELOS = ["Motor porquinho", "Cabine", "Coletor", "Luminária", "Outros"]

# ================= CADASTRO =================
with st.form("form_os"):
    st.header("Cadastro de OS")

    Loja = st.selectbox("Loja", LOJAS)
    Cliente = st.text_input("Cliente")
    Equipamento = st.selectbox("Equipamento", EQUIPAMENTOS)
    Marca = st.selectbox("Marca", MARCAS)
    Modelo = st.selectbox("Modelo", MODELOS)
    Defeito = st.text_area("Defeito")
    Data_entrada = st.date_input("Data de entrada")

    submitted = st.form_submit_button("Cadastrar")

# FORA DO FORM
if submitted:
    if not Cliente:
        st.error("Digite o cliente!")
    else:
        dados = (
            Loja,
            Cliente,
            Equipamento,
            Marca,
            Modelo,
            Defeito,
            Data_entrada.strftime("%d/%m/%Y"),
            None,
            ""
        )

        cursor.execute('''
            INSERT INTO ordens
            (LOJA, CLIENTE, EQUIPAMENTO, MARCA, MODELO, DEFEITO, DATA_ENTRADA, DATA_REPARO, MES_REPARO)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
        ''', dados)

        conn.commit()

        os_id = cursor.lastrowid  # ✔ agora alinhado corretamente

        cursor.execute("SELECT * FROM ordens WHERE id=?", (os_id,))
        os_data = cursor.fetchone()

        pdf = gerar_pdf(os_data)

        st.success(f"OS {os_id} criada!")

        with open(pdf, "rb") as f:
            st.download_button("📄 Baixar PDF", f, file_name=os.path.basename(pdf))

# ================= LISTAGEM =================
st.header("📊 Ordens de serviço")

cursor.execute("SELECT * FROM ordens")
dados = cursor.fetchall()

if dados:
    colunas = [desc[0] for desc in cursor.description]
    df_os = pd.DataFrame(dados, columns=colunas)

    # ================= EDITAR =================
    st.subheader("✏️ Editar dados")

    df_editado = st.data_editor(
        df_os,
        column_config={"id": st.column_config.NumberColumn(disabled=True)},
        use_container_width=True
    )

    if st.button("💾 Salvar alterações"):
        for _, row in df_editado.iterrows():
            cursor.execute(textwrap.dedent('''
                    UPDATE ordens
                    SET LOJA=?, CLIENTE=?, EQUIPAMENTO=?, MARCA=?, MODELO=?, DEFEITO=?, DATA_ENTRADA=?, DATA_REPARO=?, MES_REPARO=?
                    WHERE id=?
                '''),
                (
                    upper_safe(row["loja"]),upper_safe(row["cliente"]),
                    upper_safe(row["equipamento"]),upper_safe(row["marca"]),
                    upper_safe(row["modelo"]),upper_safe(row["defeito"]),row["data_entrada"],
                    row["data_reparo"],upper_safe(row["mes_reparo"]),row["id"]
                )
            )

        conn.commit()
        st.success("Alterações salvas!")
        st.rerun()

    # ================= EXPORTAR =================
    st.subheader("Exportar dados")

    def exportar_excel():
        buffer = io.BytesIO()
        with pd.ExcelWriter(buffer, engine='openpyxl') as writer:
            df_os.to_excel(writer, index=False)
        return buffer.getvalue()

    st.download_button("📥 Excel", exportar_excel(), "ordens.xlsx")

    # PDF
    def exportar_pdf(df):
        buffer = io.BytesIO()
        c = canvas.Canvas(buffer, pagesize=A4)
        c.setFont("Times-Roman", 12)

        y = 800
        for _, row in df.iterrows():
            texto = f"OS {row['id']} | {row['cliente']} | {row['equipamento']}"
            c.drawString(50, y, texto)
            y -= 20

            if y < 50:
                c.showPage()
                y = 800

        c.save()
        buffer.seek(0)
        return buffer

    st.download_button("📄 PDF", exportar_pdf(df_os), "ordens.pdf")

    # ================= EXCLUIR =================
    st.subheader("🗑️ Excluir OS")

    os_del = st.selectbox("Selecione", df_os["id"])

    if st.button("Excluir"):
        cursor.execute("DELETE FROM ordens WHERE id=?", (os_del,))
        conn.commit()
        st.warning(f"OS {os_del} excluída!")
        st.rerun()

    # ================= FINALIZAR =================
    st.subheader("Finalizar OS")

    for row in dados:
        col1, col2, col3 = st.columns([4,2,2])

        with col1:
            st.write(f"OS {row[0]} - {row[2]}")

        with col2:
            st.write("🟢 Finalizado" if row[8] else "🟡 Pendente")

        with col3:
            if not row[8]:
                if st.button("Finalizar", key=row[0]):
                    finalizar_os(row[0])
                    st.rerun()

else:
    st.info("Nenhuma OS encontrada")
"""
with open("app.py", "w") as f:
    f.write(streamlit_code)
# ================= NGROK =================
from pyngrok import ngrok
from pyngrok.exception import PyngrokNgrokHTTPError
import subprocess
import time

# Kill any ngrok processes already running to prevent 'endpoint already online' errors
# More aggressive kill command
subprocess.run(['killall', '-9', 'ngrok'], stderr=subprocess.DEVNULL, stdout=subprocess.DEVNULL)
ngrok.kill()

# Set your ngrok authtoken. You can get one from https://dashboard.ngrok.com/get-started/your-authtoken
# You can add it to Colab's secrets management (left panel, key icon) as 'NGROK_AUTH_TOKEN'
from google.colab import userdata
NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN') # <-- Descomentado e configurado
ngrok.set_auth_token(NGROK_AUTH_TOKEN) # <-- Descomentado e configurado

# Start ngrok tunnel with retry logic
public_url = None
max_retries = 5
initial_delay = 5 # seconds

for i in range(max_retries):
    try:
        # Ensure any existing ngrok process is terminated first
        subprocess.run(['killall', '-9', 'ngrok'], stderr=subprocess.DEVNULL, stdout=subprocess.DEVNULL)
        ngrok.kill()
        time.sleep(initial_delay) # Wait a bit after killing

        public_url = ngrok.connect(8501)
        print(f"Streamlit App URL: {public_url}")
        break # Exit loop if successful
    except PyngrokNgrokHTTPError as e:
        if "endpoint is already online" in str(e) or "ngrok client exception" in str(e):
            print(f"Encountered ngrok error: {e}. Attempt {i+1}/{max_retries}. Retrying in {initial_delay * (i+1)} seconds...")
            time.sleep(initial_delay * (i+1)) # Increasing sleep time
        else:
            raise e # Re-raise if it's a different type of ngrok error

if not public_url:
    raise Exception("Failed to establish ngrok tunnel after multiple retries.")

# ================= RODAR =================
import subprocess
subprocess.Popen([
    "streamlit", "run", "app.py",
    "--server.port", "8501",
    "--server.headless", "true"
])

Streamlit App URL: NgrokTunnel: "https://facebook-enticing-aneurism.ngrok-free.dev" -> "http://localhost:8501"


<Popen: returncode: None args: ['streamlit', 'run', 'app.py', '--server.port...>

In [ ]:
pip install reportlab streamlit pyngrok openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 65.9 MB/s eta 0:00:00
